In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [2]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.transforms import v2
from datasets import load_dataset
from tqdm import tqdm

from src.utils import get_backbone
from src.models import Encoder

/Users/den/Desktop/research/jepas/lejepa/lejepa-sigreg-ssl/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")

In [4]:
ckpt = torch.load("../ckpts/checkpoint_epoch_399.pt", map_location=device)

backbone = get_backbone("vit", "tiny")
encoder = Encoder(backbone, dim=192, proj_dim=32).to(device)

encoder.load_state_dict(ckpt["encoder"])

<All keys matched successfully>

Custom dataset on top of huggingface dataset

In [5]:
class HFDataset(torch.utils.data.Dataset):
    def __init__(self, dataset_name="frgfm/imagenette", split="train", V=1):
        self.ds = load_dataset(dataset_name, "160px", split=split)
        self.V = V
        self.aug = v2.Compose(
            [
                v2.Resize(128),
                v2.CenterCrop(128),
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ]
        )

    def __getitem__(self, i):
        item = self.ds[i]
        img = item["image"].convert("RGB")

        img_tensor = self.aug(img)
        views = torch.stack([img_tensor for _ in range(self.V)])

        return views, item["label"]

    def __len__(self):
        return len(self.ds)

In [6]:
train_ds = HFDataset()
val_ds = HFDataset(split="validation")

In [7]:
train_loader = DataLoader(train_ds, batch_size=32)
val_loader = DataLoader(val_ds, batch_size=32)

Extract embeddings from a trained encoder to get "frozen" features

In [8]:
def load_embeddings(encoder, loader, device):
    X, y = [], []

    encoder.eval()
    with torch.no_grad():
        for xs, ys in tqdm(loader):
            xs, ys = xs.to(device), ys.to(device)

            with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
                embs, _ = encoder(xs)

            X.append(embs.detach())
            y.append(ys.detach())

    X, y = torch.cat(X, dim=0), torch.cat(y, dim=0)
    return X, y

In [9]:
X_train, y_train = load_embeddings(encoder, train_loader, device)
X_test, y_test = load_embeddings(encoder, val_loader, device)

100%|██████████| 123/123 [00:20<00:00,  6.06it/s]


In [10]:
def knn_predict(X_train, X_test, y_train, k=3):
    distances = torch.cdist(X_train, X_test)
    nearest = torch.topk(distances, k=k, dim=0, largest=False)[1] # take only indices
    classes = y_train[nearest]
    y_pred, _ = torch.max(classes, dim=0)
    return y_pred

In [13]:
for i in [1, 3, 5, 10, 20, 50]:
    y_pred = knn_predict(X_train, X_test, y_train, k=i)
    accuracy = (y_pred == y_test).float().mean().item()
    print("KNN probing with k=", i, "is", accuracy*100)

KNN probing with k= 1 is 76.38216614723206
KNN probing with k= 3 is 73.75796437263489
KNN probing with k= 5 is 70.21656036376953
KNN probing with k= 10 is 62.42038011550903
KNN probing with k= 20 is 53.65605354309082
KNN probing with k= 50 is 38.394904136657715
